# 📈 burst_scan — the move study: can any metric predict a big move, and which way?

The burst study asked *what happens after a burst*. This one turns it around: the **event is the outcome a trade
needs** — a move beyond a barrier within 4 hours — and every name is checked every 30 minutes. Two questions, in
order:

1. **Occurrence** — do any metrics predict *that* a big move is coming? (Context: expected to work, because
   volatility clusters; the test is whether they add anything beyond the stock's own normal range.)
2. **Direction** — among moments where a move comes, do any metrics predict *which way*? (The verdict.)

| § | what |
|---|---|
| 1 | load the 5-minute panel, the decision points and the costs |
| 2 | **base rates** — how often a move comes, by volatility, time of day and name; fixed 2% vs scaled |
| 3 | **the barrier** — `k` chosen by the pre-registered rule (20% of training points reach it) |
| 4 | occurrence: each metric alone (within-month AUC) |
| 5 | occurrence: the walk-forward model against the "normal range + time of day" baseline |
| 6 | direction: each metric alone |
| 7 | **direction: the walk-forward model**, picks inside stage 1's top third, controls, monthly stability |
| 8 | **verdict** per side (scaled barrier, and the fixed-2% variant), checks, scorecard |

The pre-registration is in the README ("Move study"). Everything is on the training window; the holdout stays
unread. AUC is always **within-month** (the mean of the monthly AUCs), so a base rate that drifts from month to
month cannot pose as skill.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import importlib.util
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_simple import backtest as bt, plots, strategies
from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
# research.py holds every computation the five notebooks share; signals() never imports it.
_spec = importlib.util.spec_from_file_location(
    "burst_research", strategies.STRATEGIES_DIR / "burst_scan" / "research.py"
)
rs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rs)
strat, S = rs.STRAT, rs.S  # the loaded strategy and its module's functions
compose = S["compose"]
SIDES = ("long", "short")
COLOR = {"long": plots._LONG, "short": plots._SHORT}


def below(fig, y=-0.1):
    """Legend under the plot: the default top-right legend sits on the subplot titles."""
    fig.update_layout(
        legend={"orientation": "h", "x": 0, "xanchor": "left", "y": y, "yanchor": "top"}
    )
    return fig


SUBSET = None  # e.g. 40 for a quick first run; None = all 200 names
OVERRIDES = {}  # any signals() knob (windows, floors, warm-up)
PRM = {**strat.params, **OVERRIDES}
SEED = 0
EVERY = 30  # minutes between decision points
FIXED = 0.02  # the fixed-barrier variant: ±2%
FEATS = [*S["FEATURES"], *rs.EXTRA_FEATURES]
WF = {
    "trade_q": 1 / 3,
    "warmup_sessions": PRM["warmup_sessions"],
    "min_train_sessions": PRM["min_train_sessions"],
}

## 1 · Panel, decision points, costs

Decision points sit every 30 minutes inside the strategy's tradable window (15 minutes after the open to 15 before
the close, after the 20-session warm-up, above the price and liquidity floors). A trade from a point enters at the
next bar's open; its barriers are measured from that bar's close, exactly as the engine books a stop and target.

In [ ]:
panel = rs.load(settings, engine, subset=SUBSET)
ctx = S["context"](panel, **PRM)
rows, cols = rs.decision_points(ctx, EVERY)
SCORE_FROM = pd.Timestamp(strat.optuna["score_from"], tz=panel.tz)
pre = ctx["index"][rows] < SCORE_FROM  # before the first out-of-sample month
_, FIRST = S["month_starts"](ctx["index"], ctx["tz"])


def mname(m):
    """A month code as YYYY-MM."""
    return f"{ctx['index'][FIRST[m]].tz_convert(ctx['tz']):%Y-%m}"


fees, slip = bt.strategy_costs(strat, panel, settings)
RT = 2 * (fees + slip)
print(
    f"panel: {panel.close.shape[0]:,} {panel.bar} bars x {len(panel.symbols)} names; "
    f"{len(rows):,} decision points ({int(pre.sum()):,} before {SCORE_FROM:%Y-%m-%d}); "
    f"round trip {1e4 * RT:.1f} bp"
)

## 2 · Base rates (Fig 2)

How often a move reaches ±2% (left) or the scaled barrier (right, after §3 picks `k`) within 4 hours, split by the
name's normal daily range. A fixed 2% is routine for a volatile name and rare for a calm one, which is why the
event threshold is scaled.

In [ ]:
FT = rs.move_table(panel, ctx, rows, cols, np.full(len(rows), FIXED))  # the fixed-2% variant
vol_q = pd.qcut(FT["atr_pct"], 5, labels=["calmest", "2", "3", "4", "wildest"])
hour = (FT["tod"] // 60).astype(int)


def rates(t, by):
    ok = t["complete"]
    g = t[ok].groupby(by[ok], observed=True)["reason"]
    return pd.DataFrame(
        {
            "up first": g.apply(lambda r: r.eq(1).mean()),
            "down first": g.apply(lambda r: r.eq(-1).mean()),
            "neither": g.apply(lambda r: r.eq(0).mean()),
            "points": g.size(),
        }
    )


base_fixed = rates(FT, vol_q)
display(base_fixed.round(3))
print("fixed ±2%, by hour after the open:")
display(rates(FT, hour).round(3).T)

## 3 · The barrier: `k` by the pre-registered rule (Fig 3)

The smallest `k` on the grid 0.5 … 4.0 whose barrier (`k` × session ATR × √(240/390)) is reached by at most 20% of
the decision points **before** the first out-of-sample month. Chosen once, before any model is fit.

In [ ]:
K, ktab = rs.calibrate_k(panel, ctx, rows[pre], cols[pre], seed=SEED)
T = rs.move_table(panel, ctx, rows, cols, rs.scaled_barrier(ctx, rows, cols, K))
lo_, mid_, hi_ = (1e2 * T["frac"].quantile(q) for q in (0.1, 0.5, 0.9))
print(
    f"k = {K:g}: barrier = {K:g} × session ATR × √(240/390) — median {mid_:.2f}% of price "
    f"(10th–90th pct {lo_:.2f}–{hi_:.2f}%)"
)
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("reach rate by k (training points)", "scaled barrier: outcome by volatility"),
)
fig.add_trace(
    go.Scatter(
        x=ktab["k"],
        y=ktab["reach"],
        mode="lines+markers",
        name="reach rate",
        line={"color": plots.SERIES[4]},
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=[ktab["k"].min(), ktab["k"].max()],
        y=[rs.MOVE_TARGET_RATE] * 2,
        mode="lines",
        name="target 20%",
        line={"color": plots._MUTED, "dash": "dash"},
    ),
    row=1,
    col=1,
)
base_scaled = rates(T, pd.qcut(T["atr_pct"], 5, labels=["calmest", "2", "3", "4", "wildest"]))
for name, colr in (("up first", plots._LONG), ("down first", plots._SHORT), ("neither", "#cfcdc4")):
    fig.add_trace(
        go.Bar(x=base_scaled.index.astype(str), y=base_scaled[name], name=name, marker_color=colr),
        row=1,
        col=2,
    )
fig.update_layout(barmode="stack")
plots._base(fig, title="Fig 3 · the pre-registered barrier", height=400)
below(fig, -0.15)
fig.update_yaxes(tickformat=".0%")
fig.show()
display(base_scaled.round(3))

## 4 · Occurrence: each metric alone (Fig 4)

Within-month AUC of each feature, taken alone, for "either barrier reached within 4 hours". 0.5 = no information;
the bar shows how far each metric is from it (either way). Descriptive, on the whole training window.

In [ ]:
done = T["complete"].to_numpy()
y_occ = np.where(done, T["reason"].ne(0), np.nan)
occ_m = rs.metric_auc(T[done], y_occ[done], FEATS)
fig = go.Figure(
    go.Bar(x=occ_m["auc"] - 0.5, y=occ_m["feature"], orientation="h", marker_color=plots.SERIES[4])
)
plots._base(fig, title="Fig 4 · occurrence: within-month AUC − 0.5, each metric alone", height=560)
fig.update_yaxes(autorange="reversed")
fig.show()
occ_m.round(3).head(10)

## 5 · Occurrence: the model against volatility (Fig 5)

The purged monthly walk-forward on all 22 features, against a **baseline that knows only the name's normal range
and the time of day**. If the full model does not beat it — paired, month by month — "predicting a move" is just
knowing which stocks are volatile.

In [ ]:
def wf(X, y, model="lr", **kw):
    return S["walk_forward"](
        np.asarray(X, float),
        np.asarray(y, float),
        T["row"].to_numpy(),
        T["exit_row"].to_numpy(),
        done,
        ctx,
        model=model,
        **{**WF, **kw},
    )


occ = wf(T[FEATS], y_occ)
occ_base = wf(T[["atr_pct", "tod"]], y_occ)
oos = occ["p"].notna().to_numpy() & done
by_month = pd.DataFrame(
    {
        "model": [
            rs.auc(occ["p"][oos & (T["month"] == m)], y_occ[oos & (T["month"] == m)])
            for m in sorted(T.loc[oos, "month"].unique())
        ],
        "baseline": [
            rs.auc(occ_base["p"][oos & (T["month"] == m)], y_occ[oos & (T["month"] == m)])
            for m in sorted(T.loc[oos, "month"].unique())
        ],
    }
)
gain = by_month["model"] - by_month["baseline"]
gain_t = gain.mean() / gain.std() * np.sqrt(len(gain)) if gain.std() > 0 else np.nan
m_auc, b_auc = by_month["model"].mean(), by_month["baseline"].mean()
print(
    f"occurrence, within-month AUC: model {m_auc:.3f} vs baseline {b_auc:.3f}; "
    f"gain {gain.mean():+.3f} (t {gain_t:.1f} over {len(gain)} months)"
)
cal = rs.calibration(occ["p"][oos], y_occ[oos])
fig = make_subplots(rows=1, cols=2, subplot_titles=("AUC by month", "calibration (out of sample)"))
fig.add_trace(
    go.Scatter(y=by_month["model"], name="all 22 metrics", line={"color": plots.SERIES[4]}),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(y=by_month["baseline"], name="normal range + time", line={"color": plots._MUTED}),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=cal["p"],
        y=cal["realized"],
        mode="lines+markers",
        name="realized",
        line={"color": plots.SERIES[4]},
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=[0, cal["p"].max()],
        y=[0, cal["p"].max()],
        mode="lines",
        name="perfect",
        line={"color": plots._MUTED, "dash": "dash"},
    ),
    row=1,
    col=2,
)
plots._base(fig, title="Fig 5 · occurrence model vs the volatility baseline", height=380)
below(fig, -0.15)
fig.show()

## 6 · Direction: each metric alone (Fig 6)

Among points where a move came, the within-month AUC of each feature for "up first" — and, for every point, the
rank correlation of the feature with the signed 4-hour return. 0.5 / 0 = no information.

In [ ]:
moved = done & T["reason"].ne(0).to_numpy()
y_up = np.where(moved, T["reason"].eq(1), np.nan)
y_dn = np.where(moved, T["reason"].eq(-1), np.nan)
dir_m = rs.metric_auc(T[moved], y_up[moved], FEATS)
ic = {
    f: T[done].groupby("month").apply(lambda g, f=f: g[f].corr(g["ret"], method="spearman")).mean()
    for f in FEATS
}
dir_m["rank IC (return)"] = dir_m["feature"].map(ic)
fig = go.Figure(
    go.Bar(x=dir_m["auc"] - 0.5, y=dir_m["feature"], orientation="h", marker_color=plots._LONG)
)
plots._base(
    fig,
    title="Fig 6 · direction: within-month AUC − 0.5 for 'up first', each metric alone",
    height=560,
)
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(range=[-0.1, 0.1])
fig.show()
dir_m.round(4).head(10)

## 7 · Direction: the model, inside stage 1's selection (Fig 7)

Two walk-forward models trained only on points that moved: P(up first) and P(down first). A **long pick** is a point
stage 1 puts in its top third ("a move is coming") where P(up first) is in its fold's top third; a **short pick**
the same with P(down first). P&L is the trade the engine would book (barrier or time exit), signed as positioned,
market not removed; costs are one round trip at the real order size.

In [ ]:
def stage2(t, occ_wf, *, model="lr", seed=SEED):
    """Direction picks inside the occurrence model's selection, per side, with every gate input."""
    d = t["complete"].to_numpy()
    mv = d & t["reason"].ne(0).to_numpy()
    X = t[FEATS].to_numpy(float)
    rws, ex = t["row"].to_numpy(), t["exit_row"].to_numpy()
    sel = occ_wf["picked"].to_numpy() & occ_wf["p"].notna().to_numpy() & d
    res = {}
    for side, lab, sign in (("long", 1, 1.0), ("short", -1, -1.0)):
        y = np.where(mv, t["reason"].eq(lab), np.nan)
        w = S["walk_forward"](X, y, rws, ex, d, ctx, model=model, **WF)
        pick = sel & w["picked"].to_numpy() & w["p"].notna().to_numpy()
        pnl = sign * t["ret"].to_numpy()
        res[side] = {"w": w, "pick": pick, "sel": sel, "pnl": pnl, "y": y, "lab": lab, "moved": mv}
    return res


def gates(t, r, *, t_min=3.0, seed=SEED):
    """Gates a–e for one side (f is decided across models)."""
    pick, sel, pnl, mv, day = r["pick"], r["sel"], r["pnl"], r["moved"], t["session"].to_numpy()
    hit = (t["reason"].to_numpy() == r["lab"]).astype(float)
    hm = rs.day_t(hit[pick & mv] - 0.5, day[pick & mv])
    net2 = np.nanmean(pnl[pick]) - 2 * RT
    a = bool(hm["mean"] > 0 and hm["t"] >= t_min and net2 > 0)
    vs_sel = rs.paired_day_t(pnl[pick], day[pick], pnl[sel], day[sel])
    rng = np.random.default_rng(seed)
    idx = np.flatnonzero(sel)
    rand = np.zeros(len(t), bool)
    rand[rng.choice(idx, size=min(int(pick.sum()), len(idx)), replace=False)] = True
    vs_rand = rs.paired_day_t(pnl[pick], day[pick], pnl[rand], day[rand])
    b = bool(vs_sel["t"] >= 2 and vs_rand["t"] >= 2)
    monthly = pd.Series(pnl[pick] - RT).groupby(t["month"].to_numpy()[pick]).mean()
    c = bool(len(monthly) and (monthly > 0).sum() >= int(np.ceil(0.7 * len(monthly))))
    by_name = pd.Series(pnl[pick]).groupby(t["symbol"].to_numpy()[pick]).sum()
    rest = ~np.isin(t["symbol"].to_numpy(), by_name.nlargest(5).index) & pick
    dd = bool(np.nanmean(pnl[rest]) - RT > 0)
    # e: outcomes permuted among moved points; scored against the permuted outcomes
    perm = rng.permutation(np.flatnonzero(mv))
    y_p = r["y"].copy()
    y_p[np.flatnonzero(mv)] = r["y"][perm]
    pnl_p = pnl.copy()
    pnl_p[np.flatnonzero(mv)] = pnl[perm]
    w_p = S["walk_forward"](
        t[FEATS].to_numpy(float),
        y_p,
        t["row"].to_numpy(),
        t["exit_row"].to_numpy(),
        t["complete"].to_numpy(),
        ctx,
        model="lr",
        **WF,
    )
    o = w_p["p"].notna().to_numpy() & mv
    sh_auc = rs.monthly_auc(w_p["p"][o], y_p[o], t["month"].to_numpy()[o])
    pk_p = sel & w_p["picked"].to_numpy() & w_p["p"].notna().to_numpy()
    sh = rs.paired_day_t(pnl_p[pk_p], day[pk_p], pnl_p[sel], day[sel])
    tol = max(0.01, 3 * rs.auc_null_se(r["y"][o]))
    e = bool(abs(sh_auc - 0.5) <= tol and (not np.isfinite(sh["t"]) or abs(sh["t"]) < 2))
    o2 = r["w"]["p"].notna().to_numpy() & mv
    wm_auc = rs.monthly_auc(r["w"]["p"][o2], r["y"][o2], t["month"].to_numpy()[o2])
    return {
        "picks": int(pick.sum()),
        "within-month auc": wm_auc,
        "hit rate (moved)": 0.5 + hm["mean"],
        "hit t": hm["t"],
        "gross bp": 1e4 * np.nanmean(pnl[pick]),
        "net 2x bp": 1e4 * net2,
        "vs selection t": vs_sel["t"],
        "vs random t": vs_rand["t"],
        "months positive": f"{int((monthly > 0).sum())}/{len(monthly)}",
        "shuffled auc": sh_auc,
        "shuffled t": sh["t"],
        "a": a,
        "b": b,
        "c": c,
        "d": dd,
        "e": e,
    }


S2 = {m: stage2(T, occ, model=m) for m in ("lr", "gbm")}
res = {}
for side in SIDES:
    g_lr = gates(T, S2["lr"][side])
    f_pair = rs.paired_day_t(
        S2["gbm"][side]["pnl"][S2["gbm"][side]["pick"]],
        T["session"].to_numpy()[S2["gbm"][side]["pick"]],
        S2["lr"][side]["pnl"][S2["lr"][side]["pick"]],
        T["session"].to_numpy()[S2["lr"][side]["pick"]],
    )
    use = "gbm" if np.isfinite(f_pair["t"]) and f_pair["t"] >= 2 else "lr"
    g = gates(T, S2[use][side]) if use == "gbm" else g_lr
    res[("scaled", side)] = {**g, "model": use, "gbm − lr t": f_pair["t"]}

# the monthly picture and what the direction model's deciles say
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "picks: net P&L per trade by month (bp)",
        "P(this side first), decile vs realized",
    ),
)
for side in SIDES:
    r = S2["lr"][side]
    pk = r["pick"]
    m = pd.Series(r["pnl"][pk] - RT).groupby(T["month"].to_numpy()[pk]).mean() * 1e4
    fig.add_trace(
        go.Bar(x=[mname(x) for x in m.index], y=m.to_numpy(), name=side, marker_color=COLOR[side]),
        row=1,
        col=1,
    )
    o = r["w"]["p"].notna().to_numpy() & r["moved"]
    cal = rs.calibration(r["w"]["p"][o], r["y"][o])
    fig.add_trace(
        go.Scatter(
            x=cal["p"],
            y=cal["realized"],
            mode="lines+markers",
            name=f"{side} realized",
            line={"color": COLOR[side]},
        ),
        row=1,
        col=2,
    )
fig.add_trace(
    go.Scatter(
        x=[0.4, 0.6],
        y=[0.4, 0.6],
        mode="lines",
        name="perfect",
        line={"color": plots._MUTED, "dash": "dash"},
    ),
    row=1,
    col=2,
)
plots._base(fig, title="Fig 7 · direction picks (scaled barrier, logistic regression)", height=400)
below(fig, -0.15)
fig.show()

## 8 · Verdict (per side), the fixed-2% variant, checks

The gates of the README's pre-registration: **a** hit rate > 50% with t ≥ 3 and positive after 2× costs; **b** beats
stage 1's selection and a random pick (t ≥ 2); **c** ≥ 70% of months positive after costs; **d** positive without the
top five names; **e** the shuffled-outcome control finds nothing (else VOID); **f** boosting only if it beats logistic
regression. The fixed-2% variant runs both stages again and needs t ≥ 3.2 on (a).

In [ ]:
occ_fixed = S["walk_forward"](
    FT[FEATS].to_numpy(float),
    np.where(FT["complete"], FT["reason"].ne(0), np.nan),
    FT["row"].to_numpy(),
    FT["exit_row"].to_numpy(),
    FT["complete"].to_numpy(),
    ctx,
    model="lr",
    **WF,
)
S2f = stage2(FT, occ_fixed)
for side in SIDES:
    res[("fixed 2%", side)] = {
        **gates(FT, S2f[side], t_min=3.2),
        "model": "lr",
        "gbm − lr t": np.nan,
    }
for g in res.values():
    g["VERDICT"] = "VOID" if not g["e"] else ("PASS" if all(g[x] for x in "abcd") else "FAIL")
verdict = pd.DataFrame(res).T
display(verdict)
checks = [
    (f"{v} · {s}: verdict", res[(v, s)]["VERDICT"], "PASS", res[(v, s)]["VERDICT"] == "PASS")
    for v, s in res
]
checks.append(
    (
        "k chosen by the rule",
        K,
        "reach ≤ 20% before 2025-10",
        bool(ktab.loc[ktab["k"] == K, "reach"].iloc[0] <= rs.MOVE_TARGET_RATE),
    )
)
checks.append(
    (
        "occurrence adds beyond volatility (paired months, t ≥ 2)",
        round(gain_t, 2),
        "≥ 2",
        bool(np.isfinite(gain_t) and gain_t >= 2),
    )
)
display(rs.checks_table(checks))
rs.scorecard(
    {
        "notebook": "move_study",
        "names": len(panel.symbols),
        "points": len(T),
        "k": K,
        "barrier_median": T["frac"].median(),
        "occurrence": {
            "model_auc": by_month["model"].mean(),
            "baseline_auc": by_month["baseline"].mean(),
            "gain_t": gain_t,
        },
        "direction": {
            f"{v}|{s}": {
                x: res[(v, s)][x]
                for x in (
                    "VERDICT",
                    "picks",
                    "within-month auc",
                    "hit rate (moved)",
                    "hit t",
                    "gross bp",
                    "net 2x bp",
                    "months positive",
                )
            }
            for v, s in res
        },
    }
)